In [ ]:
#!pip install --upgrade torchao

In [72]:
%reset -s -f

In [73]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchao

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

#Hugging Face classes chosen to handle pre-trained LLMs and abstract away complex training loops
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
#Parameter-Efficient Fine-Tuning libraries.
from peft import get_peft_model, LoraConfig, TaskType
import wandb
from kaggle_secrets import UserSecretsClient

In [74]:
print(torchao.__version__)

0.17.0


In [75]:
# All files under the input directory
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# SETUP & CONFIGURATION

In [76]:
seed=42
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Global constants
TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
LABEL_MAP = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [77]:
user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WandB-API")
wandb.login(key=wandb_api_key)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

# EDA

In [78]:
TRAIN_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

In [79]:
TRAIN_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [80]:
test_df.head()

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [81]:
TRAIN_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


.describe(include='all') generates descriptive statistics. Using include='all' ensures it also summarizes object/string columns showing counts, unique values, and the most frequent value.

.info() prints a concise summary of the DataFrame, including the index dtype and columns, non-null values, and memory usage.


# DATA PREPROCESSING

## Check Null Values

In [82]:
mcq_cols = ['prompt', 'A', 'B', 'C', 'D', 'E']

TRAIN_df[mcq_cols].isna().sum()
#So there's no null value in this dataset

prompt    0
A         0
B         0
C         0
D         0
E         0
dtype: int64

## Drop rows with duplicate prompt

In [83]:
TRAIN_df['prompt'].duplicated().sum()

np.int64(242)

In [84]:
TRAIN_df[TRAIN_df['prompt'].duplicated(keep=False)].sort_values('prompt').head(3)

,id,prompt,A,B,C,D,E,answer
605,606,Choose the correct answer: What are permutatio...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,E
456,457,Choose the correct answer: What are permutatio...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,E
439,440,Choose the correct answer: What are permutatio...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,E


Keep the first occurrence of each duplicate prompt and drops the rest:

In [85]:
TRAIN_df = TRAIN_df.drop_duplicates(subset='prompt', keep='first')
TRAIN_df['prompt'].duplicated().sum()  # should now be 0

np.int64(0)

## Case normalization - Convert all the text to lowercase

In [86]:
TRAIN_df[mcq_cols] = TRAIN_df[mcq_cols].apply(lambda col: col.str.lower())
test_df[mcq_cols] = test_df[mcq_cols].apply(lambda col: col.str.lower())

In [87]:
# Display an example
print(TRAIN_df['prompt'].iloc[0])

pick the best possible answer: what is martin heidegger's view on the relationship between time and human existence? among the listed options.


## Trip possible boiler plates

In [88]:
import re
def strip_boilerplate(text):
    return re.sub(r'^(pick the best possible answer|select the most accurate option|determine the correct option|choose the correct answer|identify the correct statement)\s*:\s*', '', text)

TRAIN_df['prompt'] = TRAIN_df['prompt'].apply(strip_boilerplate)
test_df['prompt'] = test_df['prompt'].apply(strip_boilerplate)

## Encode Options (answers)

In [89]:
encoder = LabelEncoder()
y = TRAIN_df['answer']
y = encoder.fit_transform(y)

In [90]:
TRAIN_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,what is martin heidegger's view on the relatio...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...,B
1,2,what is accelerator-based light-ion fusion?,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,A
2,3,what is the term used in astrophysics to descr...,blueshifting,redshifting,reddening,whitening,yellowing,C
3,4,what is martin heidegger's view on the relatio...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...,B
4,5,what is the concept of simultaneity in einstei...,"simultaneity is relative, meaning that two eve...","simultaneity is relative, meaning that two eve...","simultaneity is absolute, meaning that two eve...",simultaneity is a concept that applies only to...,simultaneity is a concept that applies only to...,A


## Split Train and Validation Set

`stratify` is chosen over a random split to guarantee that both datasets contain the exact same proportion of A, B, C, D, and E answers, preventing class imbalance

In [91]:
# Stratified split to ensure answer distributions match
train_df, val_df = train_test_split(TRAIN_df, test_size=0.2, random_state=42, stratify=TRAIN_df['answer'])

In [92]:
train_df.shape[0]

1406

In [93]:
val_df.shape[0]

352

# METRIC EVALUATION FUNCTION

It extracts logits (raw model predictions) and labels, and calculates standard Accuracy, Macro F1-score, and iteratively calculates the MAP@3 score.

In [94]:
def calculate_metrics(eval_preds):
    logits, labels = eval_preds
    preds = np.argsort(logits, axis=-1)[:, ::-1] # Sort in descending order

    top1_preds = preds[:, 0]
    accuracy = accuracy_score(labels, top1_preds)
    f1 = f1_score(labels, top1_preds, average="macro")

    map3_score = 0.0
    for i in range(len(labels)):
        true_label = labels[i]
        for rank in range(3):
            if preds[i, rank] == true_label:
                map3_score += 1.0 / (rank + 1)
                break
    map3_score /= len(labels)
    
    return {
        "accuracy": accuracy,
        "f1_score": f1,
        "map@3": map3_score
    }

`argsort` is chosen over `max` because we need to calculate Mean Average Precision at 3 (MAP@3) which requires top 3 rankings, not just the single best answer.

# Model-1: TF-IDF

`TfidfVectorizer` is chosen over `CountVectorizer` because it penalizes highly frequent, uninformative words. `cosine_similarity` is chosen over Euclidean distance because it measures the semantic direction of the text vectors, independent of their length.

In [95]:
X = TRAIN_df.iloc[:,1:7]
X.head()

,prompt,A,B,C,D,E
0,what is martin heidegger's view on the relatio...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...
1,what is accelerator-based light-ion fusion?,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...
2,what is the term used in astrophysics to descr...,blueshifting,redshifting,reddening,whitening,yellowing
3,what is martin heidegger's view on the relatio...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...
4,what is the concept of simultaneity in einstei...,"simultaneity is relative, meaning that two eve...","simultaneity is relative, meaning that two eve...","simultaneity is absolute, meaning that two eve...",simultaneity is a concept that applies only to...,simultaneity is a concept that applies only to...


In [96]:
X.shape[0]

1758

In [97]:
X_train_tfidf, X_val_tfidf, y_train_tfidf, y_val_tfidf = train_test_split(X, y, test_size=0.2, random_state=42)

In [98]:
X_train_tfidf.shape[0]

1406

In [99]:
# Initialize vectorizer
vectorizer = TfidfVectorizer(max_df=0.95)
# Ignores words that appear in over 95% of the text (useless for discrimination)

# Fit vocabulary on all training text (prompt + all options), so every option and prompt share the same vector space
train_corpus = pd.concat([X_train_tfidf[col] for col in mcq_cols], ignore_index=True)
vectorizer.fit(train_corpus)

# Transform each column separately, for both train and val
X_train_vecs = {col: vectorizer.transform(X_train_tfidf[col]) for col in mcq_cols}
X_val_vecs = {col: vectorizer.transform(X_val_tfidf[col]) for col in mcq_cols}

## Evaluate on validation data

In [100]:
option_cols = ['A', 'B', 'C', 'D', 'E']

# Cosine similarity between each option and its prompt, per row
sims = np.zeros((X_val_tfidf.shape[0], len(option_cols)))
for i, col in enumerate(option_cols):
    row_sims = cosine_similarity(X_val_vecs['prompt'], X_val_vecs[col])
    sims[:, i] = row_sims.diagonal()

val_metrics = calculate_metrics((sims, y_val_tfidf))
val_metrics

{'accuracy': 0.17613636363636365,
 'f1_score': 0.1715455676790795,
 'map@3': 0.29734848484848464}

# HUGGING FACE TRANSFORMER DATASETS

`tokenizer` - Converts raw text into token IDs.

Padding and truncation are chosen over dynamic lengths so the outputs form perfect rectangular tensors [Batch_Size, Num_Choices, Max_Length] that PyTorch can process on the GPU.

In [101]:
class HuggingFaceMCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test
        self.options = ['A', 'B', 'C', 'D', 'E']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        
        choices_inputs = []
        for opt in self.options:
            option_text = str(row[opt])
            choices_inputs.append((prompt, option_text))
            
        # Standard tokenization structure matching [Batch_Size, Num_Choices, Max_Length]
        features = self.tokenizer(
            [text[0] for text in choices_inputs],
            [text[1] for text in choices_inputs],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        
        item = {
            "input_ids": features["input_ids"],
            "attention_mask": features["attention_mask"]
        }
        
        if not self.is_test:
            item["labels"] = torch.tensor(LABEL_MAP[row['answer']], dtype=torch.long)
            
        return item

# Data collator to enforce precise shapes for Hugging Face multi-choice pipeline execution
def mcq_data_collator(features):
    batch = {}
    batch["input_ids"] = torch.stack([f["input_ids"] for f in features])
    batch["attention_mask"] = torch.stack([f["attention_mask"] for f in features])
    if "labels" in features[0]:
        batch["labels"] = torch.stack([f["labels"] for f in features])
    return batch

`mcq_data_collator` - Takes a list of individual dictionary items and uses torch.stack to stack them into single batched tensors

# Model-2: PRETRAINED DeBERTa

**Decoding-enhanced BERT with Disentangled Attention**

Instead of treating words as single vectors, DeBERTa treats words as two separate vectors representing their content and position. This significantly improves how the model understands the context and relationship of words in a sentence.

In [102]:
deberta_ckpt = "microsoft/deberta-v3-small"
deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_ckpt)
deberta_model = AutoModelForMultipleChoice.from_pretrained(deberta_ckpt).to(DEVICE)
deberta_train_ds = HuggingFaceMCQDataset(train_df, deberta_tokenizer, max_len=128)
deberta_val_ds = HuggingFaceMCQDataset(val_df, deberta_tokenizer, max_len=128)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                 

In [103]:
lens = deberta_tokenizer(train_df['prompt'].tolist(), truncation=False)['input_ids']
print(np.percentile([len(l) for l in lens], [50, 90, 95, 99]))

[18. 30. 34. 44.]


99th percentile prompt is 45 tokens so it is safe to use max_len=128

In [104]:
NUM_EPOCHS = 3

deberta_args = TrainingArguments(
    output_dir="./deberta_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,                
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,  #Fraction of total training steps spent increasing the lr from 0 to the initial value                
    max_grad_norm=0.5,
    # per_device_train_batch_size=8,      
    # per_device_eval_batch_size=8,
    #gradient_accumulation_steps=2,   #calculates gradients for 2 smaller batches before updating weights (Effective batch size = 8 × 2 × 2 = 32)
    num_train_epochs=NUM_EPOCHS,
    metric_for_best_model='map@3',
    load_best_model_at_end=True,
    greater_is_better=True,
    # save_total_limit=2,
    report_to="wandb",
    run_name="pretrained-deberta-run",
    logging_steps=10,
)

deberta_trainer = Trainer(
    model=deberta_model,
    args=deberta_args,
    train_dataset=deberta_train_ds,
    eval_dataset=deberta_val_ds,
    data_collator=mcq_data_collator,
    compute_metrics=calculate_metrics,
)
deberta_trainer.train()
wandb.finish()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Score,Map@3
1,3.126562,3.294922,0.210227,0.207108,0.323390
2,3.222852,3.218750,0.164773,0.056585,0.330019
3,3.236523,3.218750,0.173295,0.089376,0.330966


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

eval/accuracy,█▁▁▁▁▁▁▂▁▁
eval/f1_score,█▁▁▁▁▁▁▄▁▂
eval/loss,▁ ▅ █▅▅
eval/map@3,█▁▁▁▁▁▁▁▁▁
eval/runtime,▂▁▁▁▂▆▃▃▅█
eval/samples_per_second,▇███▇▃▆▅▄▁
eval/steps_per_second,▇███▇▃▆▅▄▁
train/epoch,▂▂▃▃▃▄▅▅▆▆▃▃▃▃▁▃▃▃▁▂▃▃▂▂▂▃▃▄▄▄▅▆▆▂▂▄▄▄▅█
train/global_step,▁▂▂▂▂▃▄▄▄▅▆▆▆▁▂▃▃▂▂▂▃▂▂▃▁▃▃▄▄▅▂▃▃▄▄▅▅▆▇█
train/grad_norm,▆▇▄▂▂ ▆▂▃ ▅▅▃█▃▂▂▂▁▂▂▁
+2,...


In [105]:
# PREDICTIONS FOR DeBERTa
# Create the test dataset and dataloader for DeBERTa
test_deberta_ds = HuggingFaceMCQDataset(test_df, deberta_tokenizer, max_len=128, is_test=True)
deberta_test_loader = DataLoader(test_deberta_ds, batch_size=4, shuffle=False, collate_fn=mcq_data_collator)

# Set the model to evaluation mode
deberta_model.eval()
deberta_probs = []

# Run inference without calculating gradients
with torch.no_grad():
    for batch in deberta_test_loader:
        inputs = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = deberta_model(**inputs).logits
        # Convert logits to probabilities
        probs = F.softmax(logits, dim=-1)
        deberta_probs.append(probs.cpu().numpy())
        
# Concatenate all batches into a single numpy array
deberta_probs = np.concatenate(deberta_probs, axis=0)

# 4. Extract Top-3 space-separated string maps for output submissions
deberta_submission_predictions = []
for probs in deberta_probs:
    # Sort indices in descending order based on probability and grab the top 3
    top3_indices = np.argsort(probs)[::-1][:3]
    # Map numeric indices back to 'A', 'B', 'C', 'D', 'E'
    top3_labels = [INV_LABEL_MAP[idx] for idx in top3_indices]
    deberta_submission_predictions.append(" ".join(top3_labels))

# Kaggle submission
submission_df2 = pd.DataFrame({
    "id": test_df["id"],
    "Prediction": deberta_submission_predictions
})

# Model-3: Fine-Tuned RoBERTa + LoRA Layers

**Robustly Optimized BERT Approach**

The training method LoRA (Low-Rank Adaptation) injects tiny trainable rank-decomposition matrices into the model.

In [106]:
# TRAINING MODEL
roberta_ckpt = "roberta-base"
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_ckpt)

base_roberta_model = AutoModelForMultipleChoice.from_pretrained(roberta_ckpt)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
roberta_peft_model = get_peft_model(base_roberta_model, lora_config).to(DEVICE)

roberta_train_ds = HuggingFaceMCQDataset(train_df, roberta_tokenizer, max_len=128)
roberta_val_ds = HuggingFaceMCQDataset(val_df, roberta_tokenizer, max_len=128)

roberta_args = TrainingArguments(
    output_dir="./roberta_lora_results",
    eval_strategy="epoch",  
    save_strategy="epoch",
    learning_rate=5e-4, 
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=12,
    report_to="wandb",
    run_name="peft-roberta-lora-run",
    logging_steps=10
)

roberta_trainer = Trainer(
    model=roberta_peft_model,
    args=roberta_args,
    train_dataset=roberta_train_ds,
    eval_dataset=roberta_val_ds,
    data_collator=mcq_data_collator,
    compute_metrics=calculate_metrics,
)

roberta_trainer.train()
wandb.finish()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.pooler.dense.bias       | MISSING    | 
classifier.weight               | MISSING    | 
roberta.pooler.dense.weight     | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Score,Map@3
1,2.833825,2.830894,0.428977,0.422867,0.611742
2,2.068345,1.672300,0.718750,0.716350,0.812500
3,1.614870,1.245389,0.838068,0.835927,0.890152
4,1.152930,0.874687,0.892045,0.892121,0.924716
5,1.430787,0.635205,0.926136,0.926319,0.954072
6,0.621076,0.428509,0.963068,0.963112,0.975852
7,0.765305,0.354883,0.980114,0.979729,0.986742
8,0.857630,0.302421,0.985795,0.986001,0.991477
9,0.458235,0.301315,0.985795,0.986212,0.992898
10,0.699194,0.246215,0.991477,0.992227,0.995265


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

eval/accuracy,▁▅▆▇▇███████
eval/f1_score,▁▅▆▇▇███████
eval/loss,█▅▄▃▂▂▁▁▁▁▁▁
eval/map@3,▁▅▆▇▇███████
eval/runtime,▁▂▃▂▂▃██▃▃▃▂
eval/samples_per_second,█▇▆▇▇▆▁▁▆▆▆▆
eval/steps_per_second,█▇▆▇▇▆▁▁▆▆▆▆
train/epoch,▁▁▁▁▁▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/global_step,▁▁▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
train/grad_norm,▁▁▁▁▂▂▃▂▅▃▅▄▅▂▃▂▃▂▂▂▂▃▂▃▄▅▄▂▃▂▁▆▄▃█▂▄▁▁▅
+2,...


In [107]:
# Create the test dataset and dataloader for RoBERTa
test_roberta_ds = HuggingFaceMCQDataset(test_df, roberta_tokenizer, max_len=128, is_test=True)
roberta_test_loader = DataLoader(test_roberta_ds, batch_size=4, shuffle=False, collate_fn=mcq_data_collator)

# Set the model to evaluation mode
roberta_peft_model.eval()
roberta_probs = []

# Run inference without calculating gradients
with torch.no_grad():
    for batch in roberta_test_loader:
        inputs = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = roberta_peft_model(**inputs).logits
        # Convert logits to probabilities
        probs = F.softmax(logits, dim=-1)
        roberta_probs.append(probs.cpu().numpy())
        
# Concatenate all batches into a single numpy array
roberta_probs = np.concatenate(roberta_probs, axis=0)

# Extract Top-3 space-separated string maps for output submissions
roberta_submission_predictions = []
for probs in roberta_probs:
    # Sort indices in descending order based on probability and grab the top 3
    top3_indices = np.argsort(probs)[::-1][:3]
    # Map numeric indices back to 'A', 'B', 'C', 'D', 'E'
    top3_labels = [INV_LABEL_MAP[idx] for idx in top3_indices]
    roberta_submission_predictions.append(" ".join(top3_labels))
    
# Kaggle submission tracking file
submission_df3 = pd.DataFrame({
    "id": test_df["id"],
    "Prediction": roberta_submission_predictions
})

Softmax is chosen over Sigmoid because this is a single-label, multi-class problem (only one option is correct out of five), whereas Sigmoid is for multi-label scenarios.

# Model-4 Ensemble

In [108]:
# EVALUATE ENSEMBLE ON VALIDATION SET

val_deberta_loader = DataLoader(deberta_val_ds, batch_size=4, shuffle=False, collate_fn=mcq_data_collator)
val_roberta_loader = DataLoader(roberta_val_ds, batch_size=4, shuffle=False, collate_fn=mcq_data_collator)

deberta_model.eval()
roberta_peft_model.eval()

val_deberta_probs, val_roberta_probs, val_true_labels = [], [], []

with torch.no_grad():
    # Get DeBERTa Validation Probs
    for batch in val_deberta_loader:
        inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
        logits = deberta_model(**inputs).logits
        val_deberta_probs.append(F.softmax(logits, dim=-1).cpu().numpy())
        val_true_labels.append(batch["labels"].cpu().numpy())
        
    # Get RoBERTa Validation Probs
    for batch in val_roberta_loader:
        inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
        logits = roberta_peft_model(**inputs).logits
        val_roberta_probs.append(F.softmax(logits, dim=-1).cpu().numpy())
        
val_deberta_probs = np.concatenate(val_deberta_probs, axis=0)
val_roberta_probs = np.concatenate(val_roberta_probs, axis=0)
val_true_labels = np.concatenate(val_true_labels, axis=0)

# Calculate Metrics
val_ensemble_probs = (0.70 * val_deberta_probs) + (0.30 * val_roberta_probs)
ensemble_metrics = calculate_metrics((val_ensemble_probs, val_true_labels))

print("--- Model 4 (Ensemble) Validation Performance ---")
print(f"Validation MAP@3: {ensemble_metrics['map@3']:.4f}")
print(f"Validation Accuracy: {ensemble_metrics['accuracy']:.4f}")
print(f"Validation F1 Score: {ensemble_metrics['f1_score']:.4f}")

#Inference Sequence
test_deberta_ds = HuggingFaceMCQDataset(test_df, deberta_tokenizer, max_len=128, is_test=True)
test_roberta_ds = HuggingFaceMCQDataset(test_df, roberta_tokenizer, max_len=128, is_test=True)

deberta_test_loader = DataLoader(test_deberta_ds, batch_size=4, shuffle=False, collate_fn=mcq_data_collator)
roberta_test_loader = DataLoader(test_roberta_ds, batch_size=4, shuffle=False, collate_fn=mcq_data_collator)

deberta_probs, roberta_probs = [], []

with torch.no_grad():
    # Loop 1: Test Inference for DeBERTa
    for batch in deberta_test_loader:
        inputs = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = deberta_model(**inputs).logits
        deberta_probs.append(F.softmax(logits, dim=-1).cpu().numpy())
        
    # Loop 2: Test Inference for RoBERTa
    for batch in roberta_test_loader:
        inputs = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = roberta_peft_model(**inputs).logits
        roberta_probs.append(F.softmax(logits, dim=-1).cpu().numpy())
        
# Concatenate Test Probs
deberta_probs = np.concatenate(deberta_probs, axis=0)
roberta_probs = np.concatenate(roberta_probs, axis=0)

# Ensembling using 70/30 weights
ensemble_probs = (0.70 * deberta_probs) + (0.30 * roberta_probs)

# Format Submissions
submission_predictions = []
for probs in ensemble_probs:
    top3_indices = np.argsort(probs)[::-1][:3]
    top3_labels = [INV_LABEL_MAP[idx] for idx in top3_indices]
    submission_predictions.append(" ".join(top3_labels))

--- Model 4 (Ensemble) Validation Performance ---
Validation MAP@3: 0.9972
Validation Accuracy: 0.9943
Validation F1 Score: 0.9952


In [109]:
submission_df = pd.DataFrame({
    "id": test_df["id"],
    "Prediction": submission_predictions
})

submission_df.to_csv("submission.csv", index=False)
print("Inference completed successfully. Output saved to: submission.csv")

Inference completed successfully. Output saved to: submission.csv
